# 01 · Exploración del conjunto de datos
*Hourly Energy Consumption (PJM Interconnection)*

**Objetivo:** conocer la estructura y la calidad de los datos de las 12 regiones antes de limpiarlos.

**Entrada:** `data/raw/*.csv` (se leen con `src.data_loader`).
**Salida:** hallazgos y decisiones de limpieza para `02_limpieza.ipynb` (sección 6). Este notebook no guarda datos.

**Contenido**
1. Configuración y carga
2. Estructura general y convención horaria
3. Calidad de los datos: cobertura, duplicados, horas faltantes (con su causa) y valores sospechosos
4. Distribución y series de tiempo
5. Primeros patrones temporales
6. Hallazgos y decisiones de limpieza

---

## 1. Configuración y carga

`regiones` guarda un DataFrame por región (para revisar la calidad una por una) y `df` junta todas en formato largo (`Datetime`, `MW`, `region`) para comparar.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from src.data_loader import load_regions_dict, load_all_regions
from src.features import agregar_variables_temporales

pio.templates.default = "plotly_white" # aplica un estilo uniforme a todos los gráficos
pd.set_option("display.max_columns", None) # evita que pandas esconda columnas con ....

In [2]:
regiones = load_regions_dict()   # {region: DataFrame}
df = load_all_regions()          # un solo DataFrame en formato largo

print(f"{len(regiones)} regiones cargadas · {len(df):,} filas en total")
df.head()

12 regiones cargadas · 1,090,167 filas en total


,Datetime,MW,region
0,2004-10-01 01:00:00,12379.0,AEP
1,2004-10-01 02:00:00,11935.0,AEP
2,2004-10-01 03:00:00,11692.0,AEP
3,2004-10-01 04:00:00,11597.0,AEP
4,2004-10-01 05:00:00,11681.0,AEP


## 2. Estructura general

Se verifica que `Datetime` sea de tipo fecha (`datetime64`), que `MW` sea numérico y cuántas filas aporta cada región.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1090167 entries, 0 to 1090166
Data columns (total 3 columns):
 #   Column    Non-Null Count    Dtype         
---  ------    --------------    -----         
 0   Datetime  1090167 non-null  datetime64[us]
 1   MW        1090167 non-null  float64       
 2   region    1090167 non-null  str           
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 29.1 MB


In [4]:
df.groupby('region').size().rename('filas').to_frame()

,filas
region,
AEP,121273
COMED,66497
DAYTON,121275
DEOK,57739
DOM,116189
DUQ,119068
EKPC,45334
FE,62874
NI,58450


### 2.1 Convención horaria

En este dataset `Datetime` marca el **final** de la hora. Si las series empiezan a la 01:00 y terminan a las 00:00, es consistente con esa convención.

In [5]:
extremos = pd.DataFrame({
    'primer_registro': {n: d['Datetime'].min() for n, d in regiones.items()},
    'ultimo_registro': {n: d['Datetime'].max() for n, d in regiones.items()},
})
extremos['hora_inicio'] = extremos['primer_registro'].dt.hour
extremos['hora_fin'] = extremos['ultimo_registro'].dt.hour
extremos

,primer_registro,ultimo_registro,hora_inicio,hora_fin
AEP,2004-10-01 01:00:00,2018-08-03,1,0
COMED,2011-01-01 01:00:00,2018-08-03,1,0
DAYTON,2004-10-01 01:00:00,2018-08-03,1,0
DEOK,2012-01-01 01:00:00,2018-08-03,1,0
DOM,2005-05-01 01:00:00,2018-08-03,1,0
DUQ,2005-01-01 01:00:00,2018-08-03,1,0
EKPC,2013-06-01 01:00:00,2018-08-03,1,0
FE,2011-06-01 01:00:00,2018-08-03,1,0
NI,2004-05-01 01:00:00,2011-01-01,1,0
PJM_Load,1998-04-01 01:00:00,2002-01-01,1,0


**Convención adoptada.** El registro de las 00:00 es el consumo de las 23:00 a las 24:00 del día anterior (la hora 24 de ese día). Por eso, a partir de `02_limpieza.ipynb`, las variables de calendario (`fecha`, `hora`, `dia_semana`, `mes`, `es_festivo`) se calculan desde `Datetime − 1 h` (día operativo), con `src.features.agregar_variables_temporales`. Los datos limpios conservan `Datetime` tal como viene en la fuente.

## 3. Calidad de los datos

### 3.1 Cobertura temporal
La idea: si una región tiene datos desde el 1 de octubre de 2004 hasta el 3 de agosto de 2018, debería tener una fila por cada hora de ese intervalo. 

Para cada región se construye el calendario ideal (una fila por hora, sin saltos, entre su primera y última fecha) y se compara con las horas que realmente existen. Esta tabla alimenta el apartado *Calidad de los datos* de `docs/02_base_de_datos.md`.

| Campo | Qué calcula |
| :--- | :--- |
| **filas** | `len(d)`: filas del archivo |
| **nulos** | `d['MW'].isnull().sum()`: cuántos MW vienen vacíos |
| **duplicados** | `d.duplicated('Datetime').sum()`: filas extra. Marca la 2.ª aparición de cada fecha repetida, no la primera |
| **horas_faltantes** | `len(faltan)` |
| **cobertura_%** | Horas únicas dividido entre horas del calendario ideal, por 100. Responde: dentro del periodo de una región, ¿qué porcentaje de las horas existe de verdad? |
| **ceros_o_negativos** | Cuántos MW son $\le 0$ (imposibles en consumo) |

In [6]:
resumen = []
faltantes_por_region = {}

for nombre, d in regiones.items():
    completo = pd.date_range(d['Datetime'].min(), d['Datetime'].max(), freq='h')
    unicas = pd.DatetimeIndex(d['Datetime'].unique())
    faltan = completo.difference(unicas)
    faltantes_por_region[nombre] = faltan

    resumen.append({
        'region': nombre,
        'fecha_inicial': d['Datetime'].min(),
        'fecha_final': d['Datetime'].max(),
        'filas': len(d),
        'nulos': int(d['MW'].isnull().sum()),
        'duplicados': int(d.duplicated('Datetime').sum()),
        'horas_faltantes': len(faltan),
        'cobertura_%': round(len(unicas) / len(completo) * 100, 2),
        'ceros_o_negativos': int((d['MW'] <= 0).sum()),
    })

resumen = pd.DataFrame(resumen).set_index('region')
resumen

,fecha_inicial,fecha_final,filas,nulos,duplicados,horas_faltantes,cobertura_%,ceros_o_negativos
region,,,,,,,,
AEP,2004-10-01 01:00:00,2018-08-03,121273,0,4,27,99.98,0
COMED,2011-01-01 01:00:00,2018-08-03,66497,0,4,11,99.98,0
DAYTON,2004-10-01 01:00:00,2018-08-03,121275,0,4,25,99.98,0
DEOK,2012-01-01 01:00:00,2018-08-03,57739,0,4,9,99.98,0
DOM,2005-05-01 01:00:00,2018-08-03,116189,0,4,23,99.98,0
DUQ,2005-01-01 01:00:00,2018-08-03,119068,0,4,24,99.98,0
EKPC,2013-06-01 01:00:00,2018-08-03,45334,0,4,6,99.99,0
FE,2011-06-01 01:00:00,2018-08-03,62874,0,4,10,99.98,1
NI,2004-05-01 01:00:00,2011-01-01,58450,0,0,14,99.98,0


In [7]:
orden = resumen.sort_values('fecha_inicial')

fig = go.Figure()
for nombre, r in orden.iterrows():
    fig.add_trace(go.Scatter(
        x=[r['fecha_inicial'], r['fecha_final']], y=[nombre, nombre],
        mode='lines', line=dict(width=8, color=px.colors.qualitative.Plotly[0]),
        name=nombre, showlegend=False,
        hovertemplate='%{y}<br>%{x|%Y-%m-%d %H:%M}<extra></extra>',
    ))
fig.update_yaxes(categoryorder='array', categoryarray=list(orden.index))
fig.update_layout(title='Cobertura temporal por región', height=500)
fig.show()

**Periodo común.** Si no todas las regiones se solapan, la comparación debe hacerse por grupos. La celda calcula el periodo común de las 12 y el de las regiones que llegan hasta el final del dataset (*vigentes*).

In [8]:
inicio_comun = resumen['fecha_inicial'].max()
fin_comun = resumen['fecha_final'].min()
print(f"Periodo común a las {len(resumen)} regiones: {inicio_comun} → {fin_comun}")
print("¿Existe solape entre todas?", inicio_comun < fin_comun)

# Regiones que llegan hasta el final del dataset
corte = resumen['fecha_final'].max() - pd.Timedelta(days=7)
vigentes = resumen[resumen['fecha_final'] >= corte]
print(f"\nRegiones vigentes ({len(vigentes)}): {list(vigentes.index)}")
print(f"Periodo común de las vigentes: {vigentes['fecha_inicial'].max()} → {vigentes['fecha_final'].min()}")
print("Fuera de ese grupo:", [r for r in resumen.index if r not in vigentes.index])

Periodo común a las 12 regiones: 2013-06-01 01:00:00 → 2002-01-01 00:00:00
¿Existe solape entre todas? False

Regiones vigentes (10): ['AEP', 'COMED', 'DAYTON', 'DEOK', 'DOM', 'DUQ', 'EKPC', 'FE', 'PJME', 'PJMW']
Periodo común de las vigentes: 2013-06-01 01:00:00 → 2018-08-03 00:00:00
Fuera de ese grupo: ['NI', 'PJM_Load']


### 3.2 Duplicados

Se revisan todas las regiones. Para cada fecha repetida se mira cuántas filas hay y cuántos valores distintos de MW: si `nunique` es 1 son filas idénticas; si es mayor, hay lecturas diferentes para la misma hora y hay que decidir cómo unificarlas.

In [9]:
detalle_dups = []
for nombre, d in regiones.items():
    dups = d[d.duplicated('Datetime', keep=False)]
    if dups.empty:
        continue
    g = dups.groupby('Datetime')['MW'].agg(['count', 'nunique', 'min', 'max'])
    g.insert(0, 'region', nombre)
    detalle_dups.append(g.reset_index())

detalle_dups = pd.concat(detalle_dups, ignore_index=True)

print("Regiones con duplicados:", detalle_dups['region'].nunique(), "de", len(regiones))
print("Horas del día en que ocurren:", sorted(int(h) for h in detalle_dups['Datetime'].dt.hour.unique()))

# Una fila por fecha duplicada: en cuántas regiones aparece y si los valores difieren
detalle_dups.groupby('Datetime').agg(
    regiones=('region', 'count'),
    max_valores_distintos=('nunique', 'max'),
)

Regiones con duplicados: 10 de 12
Horas del día en que ocurren: [2]


,regiones,max_valores_distintos
Datetime,,
2014-11-02 02:00:00,10,2
2015-11-01 02:00:00,10,2
2016-11-06 02:00:00,10,2
2017-11-05 02:00:00,10,2


In [10]:
# Ejemplo de detalle para regiones
for region in regiones:
    print(detalle_dups[detalle_dups['region'] == region])

             Datetime region  count  nunique      min      max
0 2014-11-02 02:00:00    AEP      2        2  12994.0  13190.0
1 2015-11-01 02:00:00    AEP      2        2  10542.0  10785.0
2 2016-11-06 02:00:00    AEP      2        2  10964.0  11008.0
3 2017-11-05 02:00:00    AEP      2        2  10446.0  10596.0
             Datetime region  count  nunique     min     max
4 2014-11-02 02:00:00  COMED      2        2  8869.0  9184.0
5 2015-11-01 02:00:00  COMED      2        2  7923.0  8270.0
6 2016-11-06 02:00:00  COMED      2        2  7814.0  8028.0
7 2017-11-05 02:00:00  COMED      2        2  7878.0  8198.0
              Datetime  region  count  nunique     min     max
8  2014-11-02 02:00:00  DAYTON      2        2  1623.0  1634.0
9  2015-11-01 02:00:00  DAYTON      2        2  1292.0  1324.0
10 2016-11-06 02:00:00  DAYTON      2        2  1334.0  1364.0
11 2017-11-05 02:00:00  DAYTON      2        2  1331.0  1449.0
              Datetime region  count  nunique     min     max
12 

### 3.3 Horas faltantes

Se cuentan por año y región. Un hueco de una o dos horas al año es típico del cambio de horario; cientos de horas en un año indicarían un apagón de datos.

In [11]:
faltantes_anio = pd.DataFrame({
    n: pd.Series(f.year).value_counts() for n, f in faltantes_por_region.items()
}).fillna(0).astype(int).sort_index()

faltantes_anio

,AEP,COMED,DAYTON,DEOK,DOM,DUQ,EKPC,FE,NI,PJM_Load,PJME,PJMW
1998,0,0,0,0,0,0,0,0,0,2,0,0
1999,0,0,0,0,0,0,0,0,0,2,0,0
2000,0,0,0,0,0,0,0,0,0,2,0,0
2001,0,0,0,0,0,0,0,0,0,2,0,0
2002,0,0,0,0,0,0,0,0,0,0,2,2
2003,0,0,0,0,0,0,0,0,0,0,2,2
2004,1,0,1,0,0,0,0,0,1,0,2,2
2005,2,0,2,0,1,2,0,0,2,0,2,2
2006,2,0,2,0,2,2,0,0,2,0,2,2
2007,2,0,2,0,2,2,0,0,2,0,2,2


In [12]:
fig = px.imshow(faltantes_anio, text_auto='d', color_continuous_scale='Reds', aspect='auto',
                title='Horas faltantes por año y región')
fig.update_yaxes(type='category')
fig.update_traces(hovertemplate='región=%{x}<br>año=%{y}<br>horas=%{z}<extra></extra>')
fig.update_layout(height=700)
fig.show()

**Qué muestra el gráfico**

Cada celda muestra el numero de horas que faltan en esa region ese año. La escala llega hasta 3, así que cualquier celda roja intensa son tres horas en un año de 8,760. No hay manchas grandes, es decir, no hay apagones de datos. Es la conclusión más importante: la calidad en este aspecto es muy buena.

In [13]:
# ¿En qué fechas y meses caen? (ejemplo con PJME)
f = faltantes_por_region['PJME']
print("Meses en que faltan horas:", pd.Series(f.month).value_counts().sort_index().to_dict())
list(f.strftime('%Y-%m-%d %H:%M'))

Meses en que faltan horas: {3: 12, 4: 5, 10: 5, 11: 7, 12: 1}


['2002-04-07 03:00',
 '2002-10-27 02:00',
 '2003-04-06 03:00',
 '2003-10-26 02:00',
 '2004-04-04 03:00',
 '2004-10-31 02:00',
 '2005-04-03 03:00',
 '2005-10-30 02:00',
 '2006-04-02 03:00',
 '2006-10-29 02:00',
 '2007-03-11 03:00',
 '2007-11-04 02:00',
 '2008-03-09 03:00',
 '2008-11-02 02:00',
 '2009-03-08 03:00',
 '2009-11-01 02:00',
 '2010-03-14 03:00',
 '2010-11-07 02:00',
 '2010-12-10 00:00',
 '2011-03-13 03:00',
 '2011-11-06 02:00',
 '2012-03-11 03:00',
 '2012-11-04 02:00',
 '2013-03-10 03:00',
 '2013-11-03 02:00',
 '2014-03-09 03:00',
 '2015-03-08 03:00',
 '2016-03-13 03:00',
 '2017-03-12 03:00',
 '2018-03-11 03:00']

**Causa de los huecos y duplicados.** Se comprueba si cada hora faltante o repetida coincide con un cambio de horario de EE. UU. (desde 2007: 2.º domingo de marzo y 1.er domingo de noviembre; antes: 1.er domingo de abril y último de octubre). Con `Datetime` como hora-fin, se espera que la hora que no existe en primavera sea las 03:00 y que la que se repite en otoño sea las 02:00.

In [14]:
def n_domingo(anio, mes, n):
    """n-ésimo domingo del mes (n = -1: el último)."""
    dias = pd.date_range(f'{anio}-{mes:02d}-01', periods=31, freq='D')
    dom = dias[(dias.month == mes) & (dias.dayofweek == 6)]
    return dom[n - 1] if n > 0 else dom[-1]

def cambios_de_horario(anio):
    """Regla de EE. UU.: desde 2007, marzo (2.º dom.) a noviembre (1.º dom.); antes, abril a octubre."""
    if anio >= 2007:
        return n_domingo(anio, 3, 2), n_domingo(anio, 11, 1)
    return n_domingo(anio, 4, 1), n_domingo(anio, 10, -1)

def causa(ts):
    primavera, otono = cambios_de_horario(ts.year)
    if ts.normalize() == primavera and ts.hour == 3:
        return 'primavera (03:00)'
    if ts.normalize() == otono and ts.hour == 2:
        return 'otoño (02:00)'
    return 'otra'

filas = [{'region': n, 'fecha': ts, 'tipo': 'faltante', 'causa': causa(ts)}
         for n, f in faltantes_por_region.items() for ts in f]
filas += [{'region': r.region, 'fecha': r.Datetime, 'tipo': 'repetida', 'causa': causa(r.Datetime)}
          for r in detalle_dups.itertuples()]
causas = pd.DataFrame(filas)

display(causas.groupby(['tipo', 'causa']).size().unstack(fill_value=0))
causas[causas['causa'] == 'otra']      # esperado: PJME 2010-12-10 00:00 y, quizá, alguna más

causa,otoño (02:00),otra,primavera (03:00)
tipo,,,
faltante,82,9,126
repetida,40,0,0


,region,fecha,tipo,causa
13,AEP,2010-12-10 00:00:00,faltante,otra
18,AEP,2012-12-06 04:00:00,faltante,otra
22,AEP,2014-03-11 14:00:00,faltante,otra
51,DAYTON,2010-12-10 00:00:00,faltante,otra
83,DOM,2010-12-10 00:00:00,faltante,otra
107,DUQ,2010-12-10 00:00:00,faltante,otra
148,NI,2010-12-10 00:00:00,faltante,otra
175,PJME,2010-12-10 00:00:00,faltante,otra
205,PJMW,2010-12-10 00:00:00,faltante,otra


**Lectura.** En `PJME` las faltantes de primavera caen siempre a las 03:00 del día del cambio. Hasta 2013 falta también la hora de otoño (02:00), y desde 2014 esa hora aparece duplicada, lo que explica "2 faltantes por año hasta 2013 y 1 después". La única faltante de `PJME` fuera de un cambio de horario es 2010-12-10 00:00. Confirmar el resto de regiones con la tabla de arriba.

### 3.4 Valores atípicos y sospechosos

Los valores altos en consumo eléctrico suelen ser picos reales de demanda, no errores. Por eso el conteo por rango intercuartil se separa en bajos y altos, y se revisan aparte los valores más bajos de cada región (un cero o un valor muy por debajo del promedio puede ser un error de medición).

In [15]:
df.groupby('region')['MW'].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
region,,,,,,,,
AEP,121273.0,15500.0,2591.0,9581.0,13630.0,15310.0,17200.0,25695.0
COMED,66497.0,11420.0,2304.0,7237.0,9780.0,11152.0,12510.0,23753.0
DAYTON,121275.0,2038.0,393.0,982.0,1749.0,2009.0,2279.0,3746.0
DEOK,57739.0,3105.0,600.0,907.0,2687.0,3013.0,3449.0,5445.0
DOM,116189.0,10949.0,2414.0,1253.0,9322.0,10501.0,12378.0,21651.0
DUQ,119068.0,1659.0,302.0,1014.0,1444.0,1630.0,1819.0,3054.0
EKPC,45334.0,1464.0,379.0,514.0,1185.0,1386.0,1699.0,3490.0
FE,62874.0,7792.0,1331.0,0.0,6807.0,7700.0,8556.0,14032.0
NI,58450.0,11702.0,2371.0,7003.0,9954.0,11521.0,12897.0,23631.0


In [16]:
def contar_atipicos(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return pd.Series({
        'bajos': int((s < q1 - 1.5 * iqr).sum()),
        'altos': int((s > q3 + 1.5 * iqr).sum()),
    })

df.groupby('region')['MW'].apply(contar_atipicos).unstack()

,bajos,altos
region,,
AEP,0,667
COMED,0,2388
DAYTON,0,1535
DEOK,34,1166
DOM,1,2105
DUQ,0,3391
EKPC,0,732
FE,1,1183
NI,0,1816


In [17]:
# Los 3 valores más bajos de cada región
(df.sort_values('MW')
   .groupby('region')
   .head(3)
   .sort_values(['region', 'MW']))

,Datetime,MW,region
105197,2016-10-02 05:00:00,9581.0,AEP
105198,2016-10-02 06:00:00,9604.0,AEP
105196,2016-10-02 04:00:00,9623.0,AEP
151032,2014-05-25 07:00:00,7237.0,COMED
179927,2017-09-10 06:00:00,7263.0,COMED
151033,2014-05-25 08:00:00,7269.0,COMED
222435,2008-09-14 18:00:00,982.0,DAYTON
249340,2011-10-11 02:00:00,1000.0,DAYTON
222445,2008-09-15 04:00:00,1011.0,DAYTON
364316,2018-04-22 05:00:00,907.0,DEOK


**¿Aislado o sostenido?** Un error de medición suele ser un valor aislado, muy por debajo de sus horas vecinas. Un valle real es una racha de varias horas seguidas. Para los 3 valores más bajos de cada región se calcula `MW_vs_vecinos` (el valor entre el promedio de la hora anterior y la siguiente) y `MW_vs_media_region`. Un `MW_vs_vecinos` muy por debajo de 1 (por ejemplo, menor que 0.5) indica un valor aislado y sospechoso; cercano a 1, una racha baja real.

In [18]:
filas = []
for nombre, d in regiones.items():
    d = d.sort_values('Datetime').reset_index(drop=True)
    vecinos = (d['MW'].shift(1) + d['MW'].shift(-1)) / 2
    d = d.assign(region=nombre,
                 MW_vs_media_region=d['MW'] / d['MW'].mean(),
                 MW_vs_vecinos=d['MW'] / vecinos)
    filas.append(d.nsmallest(3, 'MW'))

pd.concat(filas)[['region', 'Datetime', 'MW', 'MW_vs_media_region', 'MW_vs_vecinos']].round(2)

C:\Users\lukas\AppData\Local\Temp\ipykernel_25408\514079.py:10: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  pd.concat(filas)[['region', 'Datetime', 'MW', 'MW_vs_media_region', 'MW_vs_vecinos']].round(2)


,region,Datetime,MW,MW_vs_media_region,MW_vs_vecinos
105197,AEP,2016-10-02 05:00:00,9581.0,0.62,1.00
105198,AEP,2016-10-02 06:00:00,9604.0,0.62,0.99
105196,AEP,2016-10-02 04:00:00,9623.0,0.62,0.99
29759,COMED,2014-05-25 07:00:00,7237.0,0.63,0.98
58654,COMED,2017-09-10 06:00:00,7263.0,0.64,0.99
29760,COMED,2014-05-25 08:00:00,7269.0,0.64,0.98
34665,DAYTON,2008-09-14 18:00:00,982.0,0.48,0.94
61570,DAYTON,2011-10-11 02:00:00,1000.0,0.49,0.68
34675,DAYTON,2008-09-15 04:00:00,1011.0,0.50,0.98
55271,DEOK,2018-04-22 05:00:00,907.0,0.29,0.98


**Lectura.** `FE` (0 MW) es su primer registro y se recorta en la limpieza (`MW_vs_vecinos` queda vacío al no tener hora anterior). `DOM` (1,253 MW, 2009-12-12 00:00) y `PJMW` (487 MW, 2003-05-29 00:00) son aislados. `EKPC` (514 MW, 2013-09-29) es una racha de varias horas de madrugada de un domingo, es decir, un valle real y no un error. `DEOK` queda por revisar con esta tabla: sus mínimos del 2018-04-22 son consecutivos, pero el 2017-11-05 tiene dos lecturas de la misma hora de 1,044 y 2,064 MW.

## 4. Distribución y series de tiempo

Las regiones tienen tamaños muy distintos (por ejemplo, PJME consume unas veinte veces más que DUQ), así que la escala del eje cambia de una región a otra.

In [19]:
# Se calculan los cuartiles y los atípicos de cada región (criterio 1.5 × IQR) para
# no enviar al navegador el millón de filas; el boxplot resultante es el mismo.
fig = go.Figure()
for nombre, s in df.groupby('region')['MW']:
    q1, mediana, q3 = s.quantile([0.25, 0.5, 0.75])
    iqr = q3 - q1
    limite_bajo, limite_alto = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    dentro = s[(s >= limite_bajo) & (s <= limite_alto)]
    atipicos = s[(s < limite_bajo) | (s > limite_alto)]
    fig.add_trace(go.Box(
        name=nombre, x=[nombre], q1=[q1], median=[mediana], q3=[q3],
        lowerfence=[dentro.min()], upperfence=[dentro.max()],
        y=[atipicos.tolist()], boxpoints='outliers',
        marker=dict(size=3), showlegend=False,
    ))
fig.update_layout(title='Distribución del consumo por región',
                  xaxis_title='region', yaxis_title='MW', height=500)
fig.show()

**Interpretación.** Las regiones difieren mucho en tamaño (PJME ≈ 32,080 MW de media frente a DUQ ≈ 1,659 MW), por lo que las más pequeñas casi no se distinguen a esta escala. La nube densa de puntos sobre los bigotes superiores corresponde a picos de demanda reales y se conservará. En cambio, los valores bajos aislados de `FE` (0 MW, su primer registro), `DOM` y `PJMW` son sospechosos (ver 3.4); `DEOK` se revisa con la tabla de 3.4. `EKPC` tiene un mínimo bajo pero sostenido, por lo que es un valle real. Como las regiones cubren periodos distintos, la comparación directa no es válida.

In [20]:
fig = make_subplots(rows=4, cols=3, subplot_titles=list(regiones), shared_xaxes='all')
for i, (nombre, d) in enumerate(regiones.items()):
    fig.add_trace(go.Scattergl(
        x=d['Datetime'], y=d['MW'], mode='lines', line=dict(width=0.5),
        name=nombre, showlegend=False,
    ), row=i // 3 + 1, col=i % 3 + 1)
fig.update_layout(height=900)
fig.show()

**Interpretación.** Cada serie coincide con su periodo de cobertura y no muestra huecos visibles. Todas presentan una estacionalidad anual marcada, con picos que se repiten cada año, y no se aprecia una tendencia clara a simple vista (hay que suavizar la serie para confirmarlo). Las caídas verticales aisladas en `FE`, `DOM`, `PJMW`, `DEOK` y `DAYTON` coinciden con los valores bajos del boxplot y se revisan en 3.4. `NI` termina cuando empieza `COMED` con una escala casi idéntica, lo que sugiere que podrían ser la misma zona en periodos consecutivos (se comprueba en `03_analisis.ipynb`, sección 6.2).

## 5. Primeros patrones temporales

Para que la comparación sea justa se usan solo las regiones vigentes y su periodo común (sección 3.1). Además, cada perfil se divide entre el promedio de su región: así se compara la **forma** del consumo y no su magnitud (1.2 significa 20 % por encima de lo habitual para esa región).

Las variables de calendario se calculan con el **día operativo** (`Datetime − 1 h`, ver 2.1), de modo que `hora` va de 0 (00:00–01:00) a 23 (23:00–24:00).

In [21]:
inicio_v = vigentes['fecha_inicial'].max()
fin_v = vigentes['fecha_final'].min()

df_comun = agregar_variables_temporales(
    df[df['region'].isin(vigentes.index) & df['Datetime'].between(inicio_v, fin_v)]
)
print(f"{df_comun['region'].nunique()} regiones · días operativos "
      f"{df_comun['fecha'].min():%Y-%m-%d} → {df_comun['fecha'].max():%Y-%m-%d}")

10 regiones · días operativos 2013-06-01 → 2018-08-02


In [22]:
for col, titulo in [('hora', 'Perfil por hora del día'),
                    ('dia_semana', 'Perfil por día de la semana'),
                    ('mes', 'Perfil por mes')]:
    perfil = df_comun.groupby(['region', col])['MW'].mean().unstack(0)
    fig = px.line(perfil / perfil.mean(), title=titulo,
                  labels={'value': 'Consumo relativo al promedio'}, height=400)
    fig.show()

**Perfil por hora**
- Valle de madrugada. El mínimo está entre las 3 y las 5 h, con un consumo 15 % a 20 % por debajo del promedio. PJME llega a ≈ 0.80.
- Subida de la mañana. Entre las 5 y las 9 h el consumo pasa de ≈ 0.82 a ≈ 1.0, y sigue subiendo más despacio durante el día.
- Pico de la tarde-noche. Está entre las 18 y las 19 h, entre ≈ 1.09 (AEP) y ≈ 1.14 (PJME). Luego baja.

**Perfil por día de la semana** - (0 = lunes, 6 = domingo)
- De lunes a jueves el consumo está por encima del promedio (≈ 1.01 a 1.05), con el máximo el martes en casi todas.
- El viernes baja a ≈ 1.015 y todas las líneas convergen ahí.
- El fin de semana cae, de ≈ 0.88 a ≈ 0.97 según la región, con el domingo como día más bajo.

**Perfil por mes**
La forma es doble joroba: picos en invierno (diciembre-enero) y en verano (julio-agosto), y valles en abril y octubre, cuando el clima es templado y se usa poca calefacción o aire acondicionado. Eso apunta a la temperatura como motor principal, aunque el dataset no la trae y por eso solo se puede inferir.

**Interpretación.** El consumo sigue un ciclo diario (mínimo de madrugada y pico entre las 18 y las 19 h), uno semanal (el fin de semana cae entre un 3 % y un 12 % bajo el promedio) y uno anual con dos máximos, en invierno y en verano, y valles en abril y octubre. La forma general es común a las 10 regiones, con diferencias: COMED muestra un desfase de unas horas respecto a las demás, EKPC tiene su pico diario más tardío y un fin de semana casi plano, y EKPC domina en invierno mientras PJME domina en verano. El dataset no incluye temperatura, por lo que la relación con el clima solo puede inferirse.

## 6. Hallazgos

| Tema | Hallazgo |
|---|---|
| Estructura | 12 regiones y 1,090,167 filas. `Datetime` es `datetime64` y `MW` es `float64`. Sin nulos en ninguna región. |
| Convención horaria | `Datetime` marca el **final** de la hora (hora del este de EE. UU. con cambio de horario). El registro de las 00:00 es la hora 24 del día anterior, así que el calendario se calcula desde `Datetime − 1 h` (día operativo). |
| Periodo común | **No existe un periodo común a las 12 regiones.** `PJM_Load` cubre 1998-04 → 2002-01 y `NI` 2004-05 → 2011-01. Las otras 10 comparten 2013-06-01 → 2018-08-03 (días operativos hasta el 2018-08-02). |
| Cobertura | Entre 99.98 % y 99.99 % en todas las regiones. |
| Duplicados | 4 fechas repetidas (2014-11-02, 2015-11-01, 2016-11-06 y 2017-11-05) a las 02:00 en 10 regiones (no en `NI` ni `PJM_Load`), con valores distintos. Son el primer domingo de noviembre: cambio de horario de otoño. |
| Horas faltantes | Entre 6 y 30 por región, de 1 a 3 por año. Son la hora de primavera (03:00) y, hasta 2013, la de otoño (02:00); desde 2014 la de otoño aparece duplicada. Excepción en `PJME`: 2010-12-10 00:00. Confirmar el resto de regiones con la tabla de 3.3. |
| Valores sospechosos | `FE` 0 MW es su primer registro (se recorta en la limpieza). Aislados y sin explicar: `DOM` 1,253 MW (2009-12-12) y `PJMW` 487 MW (2003-05-29), 2,553 y 2,574 MW (2012-04-01/02). Se conservan en la limpieza; su tratamiento con el z modificado queda por hacer. `EKPC` (514 MW) es una racha baja real, no un error. `DEOK`: pendiente de revisar con la tabla de 3.4. |
| Atípicos (IQR) | Altos: de 567 (`PJM_Load`) a 3,450 (`PJME`) por región, picos reales de demanda que se conservan. Bajos: 34 en `DEOK`, 10 en `PJMW`, 5 en `PJME` (2012-10-30, huracán Sandy), 1 en `DOM`, 1 en `FE` y 0 en el resto. |
| Escala | Las magnitudes difieren mucho (`PJME` ≈ 32,080 MW de media frente a `DUQ` ≈ 1,659 MW). Para comparar formas hay que normalizar. |